## ENEM 2023 — Recortes geográficos,  variáveis derivadas (pós‐pré-processamento)

Objetivo: a partir do dataset tratado (nível candidato), criar variáveis derivadas usadas no dashboard/EDA e gerar recortes geográficos (MG ), mantendo tipagem consistente e reduzindo categorias não utilizadas.

Entradas: DADOS_TRATADOS_2023 (Parquet, saída do notebook 00-fbps-preprocessamento_2023).
Saídas:

DADOS_TRATADOS_2023_MEDIAS (Parquet)  — arquivo original com as duas variáveis derivadas adicionadas

DADOS_TRAT_MG_2023 (Parquet) — candidatos com prova em MG + variável regiao

(opcional/etapa posterior) tabelas agregadas por município/região (quando você criar/rodar df_agg)

Observação (GitHub): microdados brutos não são versionados; esta etapa opera sobre Parquet já tratado.

In [1]:
import sys
from pathlib import Path

# Permite importar o pacote `src/` a partir do diretório do projeto.
ROOT_PATH = Path().resolve().parents[1]  # notebooks/00_preprocessamento -> projeto
if str(ROOT_PATH) not in sys.path:
    sys.path.append(str(ROOT_PATH))

import re
import numpy as np
import pandas as pd


from src.config import (
    DADOS_TRATADOS_2023, 
    DADOS_TRATADOS_MEDIAS_2023,
    DADOS_TRAT_MG_2023, 
)

from src.preprocessamento.regioes_mg import  atribuir_regiao, MAP_NOME_REGIAO, MAP_REGIOES
from src.preprocessamento.categorias import MAP_SAL_MIN_RENDA_MEDIA




pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")




### 2) Leitura do dataset tratado

In [2]:
df = pd.read_parquet( DADOS_TRATADOS_2023)


df.head(3)

,faixa_etaria,sexo,estado_civil,cor_raca,escola,municipio,uf,nota_cn,nota_ch,nota_lc,nota_mt,lingua,nota_redacao,escolaridade_pai,escolaridade_mae,ocup_pai,ocup_mae,n_pessoas_resd,sal_min,emp_domst,gelad,lv_rp,tv,cel,comptdr,indice_consumo
0,36-45,masculino,casado/mora com companheiro,branca,não informada,Brasília,DF,NaN,NaN,NaN,NaN,inglês,NaN,não estudou,superior,superior,médio/tec,5,1 a 3,3,2,1,1,1,0,0.49
1,26-35,masculino,casado/mora com companheiro,branca,não informada,Brasília,DF,NaN,NaN,NaN,NaN,inglês,NaN,superior,médio,superior,básico,3,3 a 5,0,1,1,1,2,3,0.32
2,20-25,feminino,solteiro,branca,não informada,Caxias do Sul,RS,502.00,498.90,475.60,363.20,espanhol,700.00,desconhecida,médio,manual/tec,desconhecida,5,1 a 3,0,1,1,1,0,0,0.18


### 3) Variáveis derivadas

Nesta etapa são criadas variáveis auxiliares que facilitam análises exploratórias, agregações e etapas posteriores do pipeline, incluindo o dashboard e a modelagem preditiva. Essas variáveis não fazem parte dos microdados originais do ENEM, mas são derivadas de campos já existentes para melhorar a interpretabilidade e a usabilidade dos dados.

#### renda_media (proxy numérica de sal_min)

A variável sal_min representa a renda familiar mensal em faixas de salários mínimos, sendo portanto uma categoria ordinal. Para permitir análises quantitativas simples — como correlações, cálculos de médias, rankings e agregações — foi criada a variável renda_media, que associa a cada faixa um valor numérico representativo, aproximando o ponto médio de cada intervalo.

Essa transformação não pretende estimar a renda real do candidato, mas apenas fornecer uma proxy numérica consistente para análises estatísticas e comparações entre grupos.

#### nota_media (indicador sintético de desempenho)

Para obter uma medida sintética do desempenho do candidato no ENEM, foi criada a variável nota_media, definida como a média aritmética das cinco notas do exame:

* Ciências da Natureza (nota_cn)
* Ciências Humanas (nota_ch)
* Linguagens (nota_lc)
* Matemática (nota_mt)
* Redação (nota_redacao)

Essa variável fornece uma visão geral do desempenho global do candidato, sendo útil para:

* comparações entre grupos socioeconômicos
* análises regionais
* visualizações no dashboard
* definição de variável alvo em experimentos de modelagem preditiva

Embora simplifique o desempenho em um único indicador, a média preserva a escala das notas individuais e facilita análises agregadas mais diretas.

##### Observação metodológica

As notas do ENEM são apresentadas em uma escala comparável entre áreas, o que permite a construção de uma medida sintética de desempenho para fins analíticos. Assim, nota_media não substitui análises específicas por área do conhecimento, mas funciona como um indicador global adequado ao objetivo deste projeto, que é investigar padrões socioeconômicos associados ao desempenho educacional.

In [3]:
#renda média

df["renda_media"] = (
    df["sal_min"].astype("string").map(MAP_SAL_MIN_RENDA_MEDIA).astype("float32")
)

In [4]:
#nota média
df["nota_media"] = df[
    ["nota_cn", "nota_ch", "nota_lc", "nota_mt", "nota_redacao"]
].mean(axis=1).astype("float32")

### 4) Recortes geográficos e limpeza de categorias

In [6]:
df_mg = df.loc[df['uf'] == 'MG', :].copy()
df_mg['uf'] = df_mg['uf'].cat.remove_unused_categories()
df_mg.head(3)

,faixa_etaria,sexo,estado_civil,cor_raca,escola,municipio,uf,nota_cn,nota_ch,nota_lc,nota_mt,lingua,nota_redacao,escolaridade_pai,escolaridade_mae,ocup_pai,ocup_mae,n_pessoas_resd,sal_min,emp_domst,gelad,lv_rp,tv,cel,comptdr,indice_consumo,renda_media,nota_media
7,26-35,masculino,solteiro,negra,não informada,Governador Valadares,MG,NaN,NaN,NaN,NaN,inglês,NaN,até fund,até fund,básico,básico,1,até 1,0,1,1,0,1,1,0.17,0.50,NaN
19,26-35,masculino,solteiro,branca,não informada,São João del Rei,MG,NaN,NaN,NaN,NaN,inglês,NaN,desconhecida,desconhecida,básico,básico,3,1 a 3,0,1,1,2,3,1,0.28,2.00,NaN
22,20-25,feminino,solteiro,branca,não informada,Alfenas,MG,707.00,667.60,628.10,816.80,inglês,820.00,até fund,pós-grad,básico,superior,4,3 a 5,0,1,1,1,3,2,0.32,4.00,727.90


### 5) Variável regiao (macro-regiões dentro de MG)


#### 5.1) Atribuição de região e mapeamento dos nomes para versões mais amigáveis



In [7]:
# Aplicar no dataframe
df_mg["regiao"] = df_mg["municipio"].astype("string").apply(atribuir_regiao)

df_mg["regiao"] = df_mg["regiao"].map(MAP_NOME_REGIAO).astype("category")
df_mg.head(3)




,faixa_etaria,sexo,estado_civil,cor_raca,escola,municipio,uf,nota_cn,nota_ch,nota_lc,nota_mt,lingua,nota_redacao,escolaridade_pai,escolaridade_mae,ocup_pai,ocup_mae,n_pessoas_resd,sal_min,emp_domst,gelad,lv_rp,tv,cel,comptdr,indice_consumo,renda_media,nota_media,regiao
7,26-35,masculino,solteiro,negra,não informada,Governador Valadares,MG,NaN,NaN,NaN,NaN,inglês,NaN,até fund,até fund,básico,básico,1,até 1,0,1,1,0,1,1,0.17,0.50,NaN,Vale do Rio Doce
19,26-35,masculino,solteiro,branca,não informada,São João del Rei,MG,NaN,NaN,NaN,NaN,inglês,NaN,desconhecida,desconhecida,básico,básico,3,1 a 3,0,1,1,2,3,1,0.28,2.00,NaN,Campo das Vertentes
22,20-25,feminino,solteiro,branca,não informada,Alfenas,MG,707.00,667.60,628.10,816.80,inglês,820.00,até fund,pós-grad,básico,superior,4,3 a 5,0,1,1,1,3,2,0.32,4.00,727.90,Sul de Minas


#### 5.2) Diagnóstico: municípios não classificados

In [8]:
todas = set()
for cidades in MAP_REGIOES.values():
    todas.update(cidades)

municipios_df = df_mg["municipio"].astype("string").str.strip().unique()
nao_classificados = [m for m in municipios_df if m not in todas]

print("Municípios únicos em MG:", len(municipios_df))
print("Não classificados:", len(nao_classificados))
# se quiser, mostre só os primeiros 30 para não poluir o notebook
print("Exemplos:", nao_classificados[:30])

Municípios únicos em MG: 188
Não classificados: 0
Exemplos: []


### Exportação
Ao final desta etapa, são salvos três arquivos:

* a base completa com variáveis derivadas (DADOS_TRATADOS_MEDIAS_2023);
* o recorte de Minas Gerais com variável regional (DADOS_TRAT_MG_2023);

Esses arquivos servem de entrada para as etapas posteriores de consolidação longitudinal, dashboard e modelagem.

In [9]:
df_mg.to_parquet(DADOS_TRAT_MG_2023, index=False)  
df.to_parquet(DADOS_TRATADOS_MEDIAS_2023, index=False)


print("✅ Salvo base completa com médias:", DADOS_TRATADOS_MEDIAS_2023, "| shape:", df.shape)
print("✅ Salvo recorte MG:", DADOS_TRAT_MG_2023, "| shape:", df_mg.shape)

✅ Salvo base completa com médias: E:\ciencias_dados\projetos\projeto_enem_ml\dados\intermediarios\dados_tratados_2023_medias.parquet | shape: (3933955, 28)
✅ Salvo recorte MG: E:\ciencias_dados\projetos\projeto_enem_ml\dados\intermediarios\dados_tratados_MG_23.parquet | shape: (358575, 29)
✅ Salvo recorte Caxambu: E:\ciencias_dados\projetos\projeto_enem_ml\dados\intermediarios\dados_tratados_CAX_23.parquet | shape: (939, 28)
